# Lecture 8: Learning from Graph Data, Link Prediction and Structural Similarity — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the first graph lecture: adjacency representations, degree distributions and power laws, local similarity scores (common neighbours, Jaccard, cosine), PageRank with teleportation, SimRank by power iteration, and how to evaluate link prediction honestly (precision/recall under class imbalance, Hits@K, MRR, AUC). The self-study used `networkx`; here every graph is a plain NumPy adjacency matrix so you implement each score yourself.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# Zachary's Karate Club (same 0-indexed edge list as networkx.karate_club_graph(), 34 nodes, 78 edges)
KARATE_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 10), (0, 11), (0, 12), (0, 13),
    (0, 17), (0, 19), (0, 21), (0, 31), (1, 2), (1, 3), (1, 7), (1, 13), (1, 17), (1, 19), (1, 21), (1, 30), (2, 3),
    (2, 7), (2, 8), (2, 9), (2, 13), (2, 27), (2, 28), (2, 32), (3, 7), (3, 12), (3, 13), (4, 6), (4, 10), (5, 6),
    (5, 10), (5, 16), (6, 16), (8, 30), (8, 32), (8, 33), (9, 33), (13, 33), (14, 32), (14, 33), (15, 32), (15, 33),
    (18, 32), (18, 33), (19, 33), (20, 32), (20, 33), (22, 32), (22, 33), (23, 25), (23, 27), (23, 29), (23, 32),
    (23, 33), (24, 25), (24, 27), (24, 31), (25, 31), (26, 29), (26, 33), (27, 33), (28, 31), (28, 33), (29, 32),
    (29, 33), (30, 32), (30, 33), (31, 32), (31, 33), (32, 33)]
# Ground-truth split after the club broke up: 0 = "Mr. Hi", 1 = "Officer"
KARATE_CLUB = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

def edges_to_adj(edges, n=None, directed=False):
    """Edge list -> dense 0/1 NumPy adjacency matrix."""
    n = n if n is not None else max(max(e) for e in edges) + 1
    A = np.zeros((n, n))
    for u, v in edges:
        A[u, v] = 1
        if not directed:
            A[v, u] = 1
    return A

A_karate = edges_to_adj(KARATE_EDGES, 34)

from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(8)

# Small undirected toy graph used in the by-hand exercises
TOY_EDGES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 4), (2, 3), (3, 5)]
A_toy = edges_to_adj(TOY_EDGES, 6)

## Part A · Concepts

### Exercise 1 · Directed, undirected, matrix or list?
*🧠 Concept · ★☆☆*

1. Give one real network that is naturally **directed** and one that is **undirected**. What is special about the adjacency matrix
   $A$ of an undirected graph, and what is $\sum_{ij} A_{ij}$ in terms of $|E|$?
2. Why do graph libraries store large sparse networks as **adjacency lists** rather than adjacency matrices? Relate your answer to
   Critical Insight 4.3 (CORA).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Follower relations are not reciprocal; friendships are. Memory of a dense matrix is $\Theta(n^2)$, of a list $\Theta(n + m)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Directed: Twitter/X follows, web hyperlinks, citations (paper → cited paper). Undirected: Facebook friendships, co-authorship.
   For an undirected graph $A = A^\top$ (each edge is stored as two reciprocal links), so $\sum_{ij}A_{ij} = 2|E|$ $= \sum_i d_i$.
2. A matrix stores all $n^2$ cells, almost all zeros in real graphs: CORA needs $2708^2\approx 7.33\times10^6$ cells, of which
   fewer than $0.15\%$ are ones. An adjacency list (like an inverted index) stores only the $2m$ neighbour entries, $\Theta(n+m)$ memory,
   and iterating a node's neighbours costs $O(d_i)$ instead of $O(n)$. (In this notebook the graphs are tiny, so dense NumPy matrices are fine.)

</details>

### Exercise 2 · Local vs global proximity
*🧠 Concept · ★☆☆*

Using the table in Critical Insight 4.1, explain in your own words:

1. **zero-similarity sparsity**: why Jaccard can't rank the pair $(4, 5)$ in the toy graph `TOY_EDGES` (draw it!) even though the nodes are
   only a few hops apart;
2. why local measures are vulnerable to **link spamming** while PageRank-style global scores are more robust;
3. the cost trade-off $O(n\langle d\rangle^2)$ vs $O(K\,n^2)$ or $O(K\,m)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$N(4) = \{1\}$, $N(5) = \{3\}$. A spammer controls the edges *from* their own pages, but not the global stationary mass.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $N(4)\cap N(5) = \emptyset$, so common neighbours, Jaccard and cosine are all exactly 0, the same as for a pair at opposite ends of the
   internet. Local scores only see 2-hop paths; $(4,5)$ is connected by paths of length 4 ($4\!-\!1\!-\!0\!-\!3\!-\!5$), which only
   global measures (random walks, Katz, SimRank, embeddings) can exploit.
2. To raise Jaccard/CN between two nodes you only need to create a few fake shared neighbours, which is cheap. Changing a stationary
   distribution requires redirecting probability flow along many paths across the whole graph, and teleportation caps how much mass a
   closed spam farm can trap.
3. Local scores touch only neighbours of neighbours: about $\langle d\rangle^2$ work per node. Global methods iterate $K$ times over the
   full matrix ($n^2$ dense) or the edge list ($m$ sparse).

</details>

### Exercise 3 · When does PageRank converge?
*🧠 Concept · ★★☆*

For each directed graph, say whether the plain random walk $q^{(t)} = q^{(t-1)}P$ converges to a unique stationary distribution from every
start, and which ergodicity condition fails otherwise:

(a) $0\to1\to2\to0$ (a directed 3-cycle). (b) Two disjoint 2-cycles $0\leftrightarrow1$, $2\leftrightarrow3$. (c) $0\to1$, $1\to0$, $1\to2$, node 2 has no out-links.
(d) Any graph after applying the Google matrix $(1-\alpha)P + \alpha U$ with $\alpha = 0.15$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Irreducible = every node reachable from every node. Aperiodic = no forced cycling through a fixed partition. A dangling node makes a zero row.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Irreducible but **periodic** (period 3): starting at node 0 the walk is at 0 only every 3rd step; $q^{(t)}$ cycles forever.
A stationary vector $(\tfrac13,\tfrac13,\tfrac13)$ exists but power iteration does not converge to it.
(b) **Not irreducible** (two components) and also periodic: infinitely many stationary distributions (any mix of the two components).
(c) Node 2 is **dangling**: its row of $P$ is all zeros, so $P$ is not row-stochastic and probability mass leaks out; $q^{(t)}\to 0$.
(d) Every entry is $\ge \alpha/n > 0$ (after dangling rows are fixed): the chain is irreducible and aperiodic, so the stationary distribution
exists, is unique, and power iteration converges from any start (geometrically, at rate $\le 1-\alpha$).

</details>

### Exercise 4 · Hiding an edge raises its Jaccard score
*🧠 Concept · ★★☆*

In the self-study, removing the true edge $(0,2)$ from the karate graph *increased* its Jaccard score from $0.2381$ to $0.2632$.
Explain exactly which sets change, and why this matters when you build a train/test split for link prediction.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$u\in N(v)$ and $v\in N(u)$ while the edge exists. Are $u$ and $v$ common neighbours of each other?

</details>

<details>
<summary><b>✅ Solution</b></summary>

While $(0,2)$ exists, $2\in N(0)$ and $0\in N(2)$. Neither is in the intersection (a node is not its own neighbour), but both are in the
union. Removing the edge leaves the intersection (5 common neighbours) unchanged and shrinks the union by 2: $5/21 = 0.2381 \to 5/19 = 0.2632$.

Consequence: scores must be computed on the **training graph with the test edges removed**; otherwise the feature "leaks" the very edge you
are trying to predict, and evaluation numbers become optimistic or simply inconsistent between positive and negative pairs.

</details>

## Part B · By hand

### Exercise 5 · Local similarity scores
*✍️ By hand · ★☆☆*

Toy graph `TOY_EDGES` $= \{(0,1),(0,2),(0,3),(1,2),(1,4),(2,3),(3,5)\}$ (undirected). For the non-edge $(1,3)$ compute

* common neighbours $|N(1)\cap N(3)|$,
* Jaccard $\frac{|N(1)\cap N(3)|}{|N(1)\cup N(3)|}$,
* cosine $\frac{|N(1)\cap N(3)|}{\sqrt{|N(1)|\,|N(3)|}}$.

In [ ]:
cn_13 = None
jac_13 = None
cos_13 = None

_n1, _n3 = set(np.flatnonzero(A_toy[1])), set(np.flatnonzero(A_toy[3]))
check('common neighbours', cn_13, len(_n1 & _n3))
check('Jaccard', jac_13, len(_n1 & _n3) / len(_n1 | _n3), tol=1e-3)
check('cosine', cos_13, len(_n1 & _n3) / np.sqrt(len(_n1) * len(_n3)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$N(1) = \{0, 2, 4\}$ and $N(3) = \{0, 2, 5\}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$N(1)\cap N(3) = \{0,2\}$ → CN $=2$. $N(1)\cup N(3) = \{0,2,4,5\}$ → Jaccard $= 2/4 = 0.5$. Cosine $= 2/\sqrt{3\cdot3} = 0.6667$.

```python
cn_13 = 2
jac_13 = 0.5
cos_13 = 2 / 3
```

</details>

### Exercise 6 · Power-law exponent from two points
*✍️ By hand · ★☆☆*

A social network's degree distribution follows $p_d = \beta d^{-\alpha}$. You measure $p_1 = 0.6$ and $p_{10} = 0.006$.

1. Use the log-linear form $\log p_d = \log\beta - \alpha\log d$ to find $\alpha$ and $\beta$.
2. Predict $p_{100}$. Is this network in the typical range $2\le\alpha\le3$?

In [ ]:
alpha_pl = None
beta_pl = None
p100 = None

_al = -np.polyfit(np.log([1, 10]), np.log([0.6, 0.006]), 1)[0]
check('alpha', alpha_pl, _al, tol=1e-3)
check('beta', beta_pl, 0.6, tol=1e-3)
check('p_100', p100, 0.6 * 100.0**(-_al), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

The slope between the two points in log-log space is $-\alpha = \frac{\log 0.006 - \log 0.6}{\log 10 - \log 1}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$-\alpha = \log_{10}(0.006/0.6) / \log_{10} 10 = \log_{10}(0.01) = -2$, so $\alpha = 2$, and $\beta = p_1 = 0.6$.
$p_{100} = 0.6\cdot 100^{-2} = 6\times10^{-5}$. Yes, $\alpha=2$ sits at the edge of the typical range: hubs are rare but far more common than
a Gaussian degree distribution would allow.

```python
alpha_pl = 2.0
beta_pl = 0.6
p100 = 0.6 * 100.0**-2
```

</details>

### Exercise 7 · Stationary distribution by hand
*✍️ By hand · ★★☆*

Directed graph: $0\to1$, $1\to2$, $2\to0$, $2\to1$. Write the row-stochastic $P$ ($P_{ij} = 1/d^{out}(i)$) and solve $q^* = q^*P$,
$\sum_i q^*_i = 1$ by hand (no teleportation). Which node is most "prestigious", and why does that make sense?

In [ ]:
q_star_hand = None   # [q0, q1, q2]

_P = np.array([[0, 1, 0], [0, 0, 1], [.5, .5, 0]])
_w, _V = np.linalg.eig(_P.T)
_v = np.real(_V[:, np.argmin(abs(_w - 1))]); _v /= _v.sum()
check('stationary distribution', q_star_hand, _v, tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

$P = \begin{pmatrix}0&1&0\\0&0&1\\\tfrac12&\tfrac12&0\end{pmatrix}$. Column $j$ of $q P$ gives $q_j$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$q_0 = \tfrac12 q_2$, $q_1 = q_0 + \tfrac12 q_2$, $q_2 = q_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

From the equations: $q_1 = \tfrac12 q_2 + \tfrac12 q_2 = q_2$ and $q_0 = \tfrac12 q_2$. Normalising, $q_2(\tfrac12 + 1 + 1) = 1$ →
$q^* = (0.2, 0.4, 0.4)$. Nodes 1 and 2 tie: node 1 receives links from 0 and half of 2's mass, and node 2 gets all of node 1's mass.
Node 0 only receives half of node 2's vote.

```python
q_star_hand = [0.2, 0.4, 0.4]
```

</details>

### Exercise 8 · The Google matrix and dangling pages
*✍️ By hand · ★★☆*

Graph with $n=4$ nodes: $0\to1$, $0\to2$, $1\to2$, $2\to0$; node 3 is **dangling** (no out-links). Teleportation $\alpha = 0.15$.

1. Compute the entries $(P_{PageRank})_{0,1}$ and $(P_{PageRank})_{0,3}$ of $(1-\alpha)P + \alpha U$.
2. What does row 3 of $(1-\alpha)P + \alpha U$ sum to? Why is that a problem, and what is the standard fix?

In [ ]:
g01 = None
g03 = None
row3_sum = None   # with the raw formula, before the fix

_P = np.zeros((4, 4)); _P[0, [1, 2]] = .5; _P[1, 2] = 1; _P[2, 0] = 1
_G = 0.85 * _P + 0.15 * np.ones((4, 4)) / 4
check('G[0,1]', g01, _G[0, 1]); check('G[0,3]', g03, _G[0, 3]); check('row 3 sum', row3_sum, _G[3].sum())

<details>
<summary><b>💡 Hint</b></summary>

$U_{ij} = 1/n = 0.25$. Row 3 of $P$ is all zeros.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $0.85\cdot\tfrac12 + 0.15\cdot\tfrac14 = 0.425 + 0.0375 = 0.4625$; $(0,3)$: $0 + 0.0375 = 0.0375$.
2. Row 3 sums to $0.15\cdot 4\cdot\tfrac14 = 0.15$, not 1: the matrix is not row-stochastic, so every step leaks $85\%$ of the mass that sits
   on node 3. The fix: replace each dangling row of $P$ by the uniform row $\tfrac1n\mathbf 1^\top$ (the surfer teleports from a dead end) *before*
   mixing with $U$. Then all rows sum to 1 and all entries are positive.

```python
g01 = 0.4625
g03 = 0.0375
row3_sum = 0.15
```

</details>

### Exercise 9 · SimRank by hand
*✍️ By hand · ★★☆*

Directed graph: `Univ`(0) $\to$ `ProfA`(1), `Univ` $\to$ `ProfB`(2), `ProfA` $\to$ `StudA`(3), `ProfB` $\to$ `StudB`(4). Decay $C = 0.8$.

Using the SimRank recursion with in-neighbour sets $I(\cdot)$, compute $s(\text{ProfA},\text{ProfB})$, $s(\text{StudA},\text{StudB})$ and
$s(\text{Univ},\text{ProfA})$. How many iterations of $R_k$ does it take to reach these values?

In [ ]:
s_profs = None
s_studs = None
s_univ_prof = None

_s = {}
_I = {0: [], 1: [0], 2: [0], 3: [1], 4: [2]}
_R = np.eye(5)
for _ in range(10):  # direct use of the definition
    _R = np.array([[1.0 if a == b else (0.0 if not _I[a] or not _I[b] else
          0.8 / (len(_I[a]) * len(_I[b])) * sum(_R[i, j] for i in _I[a] for j in _I[b])) for b in range(5)] for a in range(5)])
check('s(ProfA, ProfB)', s_profs, _R[1, 2]); check('s(StudA, StudB)', s_studs, _R[3, 4]); check('s(Univ, ProfA)', s_univ_prof, _R[0, 1])

<details>
<summary><b>💡 Hint</b></summary>

$I(\text{ProfA}) = I(\text{ProfB}) = \{\text{Univ}\}$, $I(\text{Univ}) = \emptyset$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$s(\text{ProfA},\text{ProfB}) = \frac{C}{1\cdot1}s(\text{Univ},\text{Univ}) = 0.8$ (after 1 iteration).
$s(\text{StudA},\text{StudB}) = C\,s(\text{ProfA},\text{ProfB}) = 0.64$ (after 2 iterations: similarity propagates one hop per iteration,
and $R_k$ only grows, which is the monotonicity property).
$s(\text{Univ},\text{ProfA}) = 0$ because $I(\text{Univ}) = \emptyset$.
The two students have no common neighbour at all, so Jaccard would give 0, but SimRank says they are "referenced by similar objects".

```python
s_profs = 0.8
s_studs = 0.64
s_univ_prof = 0.0
```

</details>

### Exercise 10 · Accuracy, precision and recall under class imbalance
*✍️ By hand · ★☆☆*

1. A graph has 100 nodes and 150 edges. What fraction of node pairs are edges, and what accuracy does the "always predict no link" model get?
2. From the self-study traces compute precision and recall for **seed 10** (144 predictions, 6 hits, 15 hidden edges) and for **CORA**
   (8391 candidates above threshold, 127 hits, 1055 hidden edges).
3. What is the precision of a *random* predictor on karate (78 edges, 15 hidden), choosing among all non-edges of the training graph?

In [ ]:
acc_trivial = None
prec_seed10, rec_seed10 = None, None
prec_cora, rec_cora = None, None
prec_random = None

check('trivial accuracy', acc_trivial, 1 - 150 / (100 * 99 / 2), tol=1e-4)
check('seed 10 precision', prec_seed10, 6 / 144, tol=1e-4); check('seed 10 recall', rec_seed10, 6 / 15, tol=1e-4)
check('CORA precision', prec_cora, 127 / 8391, tol=1e-4); check('CORA recall', rec_cora, 127 / 1055, tol=1e-4)
check('random precision', prec_random, 15 / (34 * 33 / 2 - (78 - 15)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Number of pairs is $\binom{n}{2}$. The training graph has $78-15 = 63$ edges; candidates are all other pairs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\binom{100}{2} = 4950$ pairs, edge rate $150/4950 = 3.03\%$, so "never a link" scores $96.97\%$ accuracy and is useless.
2. Seed 10: precision $6/144 = 4.17\%$, recall $6/15 = 40\%$. CORA: precision $127/8391 = 1.51\%$, recall $127/1055 = 12.04\%$.
3. Candidates $= 561 - 63 = 498$ pairs, 15 of which are hidden edges → random precision $15/498 = 3.01\%$.
   So seed 10's $4.17\%$ is only $1.4\times$ better than random! This is why ranking metrics (and comparing to a random baseline) matter.

```python
acc_trivial = 1 - 150 / 4950
prec_seed10, rec_seed10 = 6 / 144, 6 / 15
prec_cora, rec_cora = 127 / 8391, 127 / 1055
prec_random = 15 / 498
```

</details>

### Exercise 11 · Hits@K and MRR by hand
*✍️ By hand · ★☆☆*

For 5 query nodes, the rank of the first true (hidden) link in your recommendation list is $3, 1, 7, 2, 1$. Compute Hits@1, Hits@3 and
the MRR $=\frac{1}{|M|}\sum_i \frac{1}{\text{rank}(i)}$.

In [ ]:
hits1 = None
hits3 = None
mrr = None

_r = np.array([3, 1, 7, 2, 1])
check('Hits@1', hits1, np.mean(_r <= 1)); check('Hits@3', hits3, np.mean(_r <= 3)); check('MRR', mrr, np.mean(1 / _r), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Hits@K = fraction of queries with rank $\le K$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Hits@1 $= 2/5 = 0.4$, Hits@3 $= 4/5 = 0.8$. MRR $= (1/3 + 1 + 1/7 + 1/2 + 1)/5 = 2.9762/5 = 0.5952$.
MRR rewards putting true links near the top: moving the rank-7 query to rank 2 would raise MRR by $(0.5-0.1429)/5 = 0.071$, while Hits@3
would not change at all.

```python
hits1 = 0.4
hits3 = 0.8
mrr = (1/3 + 1 + 1/7 + 1/2 + 1) / 5
```

</details>

## Part C · Code from scratch

### Exercise 12 · Degrees and the degree distribution
*💻 Code · ★☆☆*

Using `A_karate` (34×34 NumPy adjacency matrix, built in the setup):

1. Compute the degree vector `deg`, the number of edges `m`, and the density $\frac{m}{\binom n2}$.
2. Compute the empirical degree distribution `p_d` as a dict $d\mapsto n_d/n$.
3. Which two nodes are the hubs? (They are the instructor "Mr. Hi" and the club officer.)

In [ ]:
deg = None
m = None
density = None
p_d = None
hubs = None   # the two node ids with largest degree, sorted

check('degrees sum to 2m', None if deg is None else np.sum(deg), 2 * len(KARATE_EDGES))
check('number of edges', m, len(KARATE_EDGES))
check('density', density, len(KARATE_EDGES) / (34 * 33 / 2))
check('p_d sums to 1', None if p_d is None else sum(p_d.values()), 1.0)
check('hubs', None if hubs is None else sorted(hubs), [0, 33])

<details>
<summary><b>💡 Hint</b></summary>

`deg = A.sum(1)`; `np.unique(deg, return_counts=True)`; `np.argsort(deg)[-2:]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$m = 78$, density $= 78/561 = 13.9\%$. Degrees are skewed: most nodes have 2–5 neighbours, node 33 has 17 and node 0 has 16.
Even this tiny network shows the "few hubs, many small nodes" pattern of scale-free graphs.

```python
deg = A_karate.sum(1)
m = int(A_karate.sum() / 2)
density = m / (34 * 33 / 2)
vals, counts = np.unique(deg, return_counts=True)
p_d = {int(d): c / len(deg) for d, c in zip(vals, counts)}
hubs = sorted(np.argsort(deg)[-2:].tolist())
print(m, round(density, 3), p_d, hubs)
```

</details>

### Exercise 13 · Similarity matrices with matrix products
*💻 Code · ★★☆*

Implement `similarity_matrices(A)` returning `(CN, J, COS)`, the $n\times n$ common-neighbour, Jaccard and cosine matrices, **without
Python loops**. Use $(A^2)_{ij} = |N(i)\cap N(j)|$ and $|N(i)\cup N(j)| = d_i + d_j - |N(i)\cap N(j)|$ (define 0/0 as 0).

Then reproduce the self-study's Exercise 1: Jaccard of $(0,2)$ in the karate graph before and after removing that edge
(`jac_before`, `jac_after`).

In [ ]:
def similarity_matrices(A):
    # TODO: return CN, J, COS
    return None

jac_before = None
jac_after = None

def _jac(A, u, v):
    a, b = set(np.flatnonzero(A[u])), set(np.flatnonzero(A[v]))
    return len(a & b) / len(a | b)
_A2 = A_karate.copy(); _A2[0, 2] = _A2[2, 0] = 0
check('Jaccard before', jac_before, _jac(A_karate, 0, 2), tol=1e-4)
check('Jaccard after', jac_after, _jac(_A2, 0, 2), tol=1e-4)
_res = similarity_matrices(A_karate)
if _res is not None:
    _CN, _J, _COS = _res
    _ref = np.array([[_jac(A_karate, u, v) for v in range(34)] for u in range(34)])
    check('full Jaccard matrix', _J, _ref)
    check('cosine (5, 6)', _COS[5, 6], len(set(np.flatnonzero(A_karate[5])) & set(np.flatnonzero(A_karate[6]))) / np.sqrt(A_karate[5].sum() * A_karate[6].sum()))
else:
    check('full Jaccard matrix', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

`CN = A @ A`, `d = A.sum(1)`, `union = d[:, None] + d[None, :] - CN`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.divide(CN, union, out=np.zeros_like(CN), where=union > 0)` avoids division by zero.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$5/21 = 0.2381 \to 5/19 = 0.2632$, exactly as in the self-study. The matrix product is the dense analogue of the
$O(n\langle d\rangle^2)$ neighbour-of-neighbour loop. For CORA one would use a sparse matrix (or the streaming generator from the note)
instead of materialising all $3.66\times10^6$ pairs.

```python
def similarity_matrices(A):
    CN = A @ A
    d = A.sum(1)
    union = d[:, None] + d[None, :] - CN
    J = np.divide(CN, union, out=np.zeros_like(CN), where=union > 0)
    denom = np.sqrt(np.outer(d, d))
    COS = np.divide(CN, denom, out=np.zeros_like(CN), where=denom > 0)
    return CN, J, COS

jac_before = similarity_matrices(A_karate)[1][0, 2]
_Am = A_karate.copy(); _Am[0, 2] = _Am[2, 0] = 0
jac_after = similarity_matrices(_Am)[1][0, 2]
print(jac_before, jac_after)
```

</details>

### Exercise 14 · Fitting a power law
*💻 Code · ★★☆*

Sample 100 000 degrees from a discrete power law with $\alpha = 2.5$ (`rng.zipf(2.5, size=100_000)`).
Compute the empirical $p_d$ for $d = 1,\dots,10$ and fit $\log p_d = \log\beta - \alpha\log d$ by least squares. Store the estimate in
`alpha_hat`. Why is it a bad idea to include the large-$d$ tail (e.g. $d$ up to 1000) in this regression?

In [ ]:
degs_pl = rng.zipf(2.5, size=100_000)
alpha_hat = None

check('alpha estimate', alpha_hat, 2.5, tol=0.15)

<details>
<summary><b>💡 Hint</b></summary>

`np.polyfit(np.log(d), np.log(p), 1)` returns `[slope, intercept]`; $\alpha$ is minus the slope.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The fit gives $\hat\alpha\approx2.5$. In the tail most degrees occur 0 or 1 times: $\log 0$ is undefined and single counts are extremely
noisy, yet each tail point gets the same weight in the regression as the well-estimated small-$d$ points, which biases the slope.
Better practice: logarithmic binning, fitting the complementary CDF, or the maximum-likelihood estimator
$\hat\alpha = 1 + n\left[\sum_i \ln\frac{d_i}{d_{\min}-\tfrac12}\right]^{-1}$.

```python
degs_pl = rng.zipf(2.5, size=100_000)
d = np.arange(1, 11)
p = np.array([(degs_pl == k).mean() for k in d])
alpha_hat = -np.polyfit(np.log(d), np.log(p), 1)[0]
print(alpha_hat)
```

</details>

### Exercise 15 · PageRank by power iteration
*💻 Code · ★★☆*

Implement `pagerank(A, alpha=0.15, tol=1e-12)` for a directed adjacency matrix ($A_{ij} = 1$ for $i\to j$):

1. build the row-stochastic $P$, replacing **dangling rows** by the uniform distribution;
2. form $P_{PR} = (1-\alpha)P + \alpha U$;
3. iterate $q \leftarrow qP_{PR}$ from the uniform start until $\lVert q^{(t)} - q^{(t-1)}\rVert_1 <$ `tol`.

Run it on the 5-node graph `A_pr` below and store the result in `q_pr`.

In [ ]:
A_pr = edges_to_adj([(0, 1), (0, 2), (1, 2), (1, 3), (2, 0), (3, 0), (3, 4)], 5, directed=True)   # node 4 is dangling

def pagerank(A, alpha=0.15, tol=1e-12):
    # TODO
    return None

q_pr = pagerank(A_pr)

# independent reference: principal left eigenvector of the Google matrix
_d = A_pr.sum(1, keepdims=True); _P = np.where(_d > 0, A_pr / np.maximum(_d, 1), 1 / 5)
_G = 0.85 * _P + 0.15 / 5
_w, _V = np.linalg.eig(_G.T); _v = np.real(_V[:, np.argmax(np.real(_w))]); _v /= _v.sum()
check('PageRank vector', q_pr, _v, tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

`d = A.sum(1)`; for rows with `d == 0` set `P[i] = 1/n`. The loop body is `q_new = q @ G`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Node 0 (in-links from 2 and 3, plus a share of the dangling mass) and node 2 (in-links from 0 and 1) get the most mass: $q \approx (0.325, 0.185, 0.264, 0.126, 0.100)$. The power iteration converges geometrically
with rate about $1-\alpha = 0.85$, so ~150 iterations reach $10^{-12}$. For real graphs one never forms $U$ densely: $qU = \tfrac1n\mathbf 1^\top$,
so each step is a sparse product plus a constant.

```python
A_pr = edges_to_adj([(0, 1), (0, 2), (1, 2), (1, 3), (2, 0), (3, 0), (3, 4)], 5, directed=True)

def pagerank(A, alpha=0.15, tol=1e-12):
    n = len(A)
    d = A.sum(1)
    P = np.empty_like(A, dtype=float)
    for i in range(n):
        P[i] = A[i] / d[i] if d[i] > 0 else np.full(n, 1 / n)
    G = (1 - alpha) * P + alpha / n
    q = np.full(n, 1 / n)
    while True:
        q_new = q @ G
        if np.abs(q_new - q).sum() < tol:
            return q_new
        q = q_new

q_pr = pagerank(A_pr)
print(np.round(q_pr, 4))
```

</details>

### Exercise 16 · SimRank in matrix form
*💻 Code · ★★★*

Show that the SimRank update can be written as $R_{k+1} = C\,W^\top R_k W$ followed by resetting the diagonal to 1, where
$W_{ia} = A_{ia}/|I(a)|$ (column-normalised adjacency; columns of nodes with no in-neighbours are zero). Implement
`simrank(A, C=0.8, K=10)` and store the full matrix for the Univ/Prof/Stud graph of Exercise 9 in `S_uni`, and for the **undirected** karate
graph (in-neighbours = neighbours) in `S_karate`. Also record `max_change`, the largest entry-wise change between iteration 9 and 10 on karate.

In [ ]:
A_uni = edges_to_adj([(0, 1), (0, 2), (1, 3), (2, 4)], 5, directed=True)

def simrank(A, C=0.8, K=10):
    # TODO: return the K-th iterate R_K
    return None

S_uni = simrank(A_uni)
S_karate = simrank(A_karate)
max_change = None

check('SimRank Univ graph', S_uni, [[1, 0, 0, 0, 0], [0, 1, .8, 0, 0], [0, .8, 1, 0, 0], [0, 0, 0, 1, .64], [0, 0, 0, .64, 1]])
if S_karate is not None:
    # brute-force definition on karate, 10 iterations
    _N = [np.flatnonzero(A_karate[a]) for a in range(34)]
    _R = np.eye(34)
    for _ in range(10):
        _Rn = np.array([[1.0 if a == b else 0.8 * _R[np.ix_(_N[a], _N[b])].mean() for b in range(34)] for a in range(34)])
        _R = _Rn
    check('SimRank karate', S_karate, _R)
    check('converged', max_change < 1e-2, True)
else:
    check('SimRank karate', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

$(W^\top R W)_{ab} = \sum_{i,j} W_{ia}R_{ij}W_{jb} = \frac{1}{|I(a)||I(b)|}\sum_{i\in I(a)}\sum_{j\in I(b)} R_{ij}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`indeg = A.sum(0)`; `W = A / np.where(indeg > 0, indeg, 1)` (broadcast over columns); `np.fill_diagonal(R, 1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The double sum over in-neighbour pairs is exactly a bilinear form with the column-normalised adjacency, so one iteration costs two
matrix products ($O(n^3)$ dense, $O(K\,n\,m)$-ish sparse). The changes shrink geometrically (by roughly a factor $C$ per step), so on karate
$R_{10}$ is already stable to within $10^{-2}$, matching the note's remark that SimRank stabilises after about 5 iterations.

```python
A_uni = edges_to_adj([(0, 1), (0, 2), (1, 3), (2, 4)], 5, directed=True)

def simrank(A, C=0.8, K=10, return_history=False):
    indeg = A.sum(0)
    W = A / np.where(indeg > 0, indeg, 1)[None, :]
    R = np.eye(len(A)); hist = [R]
    for _ in range(K):
        R = C * W.T @ R @ W
        np.fill_diagonal(R, 1.0)
        hist.append(R)
    return (R, hist) if return_history else R

S_uni = simrank(A_uni)
S_karate, _hist = simrank(A_karate, return_history=True)
max_change = np.abs(_hist[-1] - _hist[-2]).max()
print(np.round(S_uni, 3)); print('max change', max_change)
```

</details>

### Exercise 17 · A full link prediction experiment
*💻 Code · ★★★*

Re-implement the self-study pipeline on karate without networkx:

1. Hide 20% of the edges (15) chosen with `np.random.default_rng(seed)`; build the training adjacency `A_train`.
2. Score every pair that is **not** an edge of `A_train` with Jaccard computed on `A_train` (use Exercise 13).
3. Report `precision`, `recall` for threshold $0.2$, and the **AUC**: the probability that a random hidden edge scores higher than a random
   true non-edge (pairs not in the *original* graph), counting ties as $\tfrac12$. Implement the AUC yourself.

Write `run_link_prediction(seed)` returning `dict(precision=..., recall=..., auc=...)`; run it for seeds 10, 42, 2026.

In [ ]:
def run_link_prediction(seed, threshold=0.2):
    # TODO
    return None

results_lp = {s: run_link_prediction(s) for s in (10, 42, 2026)}

_res = results_lp.get(10)
if _res is not None:
    _r = np.random.default_rng(10)
    _idx = _r.choice(len(KARATE_EDGES), 15, replace=False)
    _hidden = [KARATE_EDGES[i] for i in _idx]
    _At = A_karate.copy()
    for u, v in _hidden: _At[u, v] = _At[v, u] = 0
    _neg = [(u, v) for u in range(34) for v in range(u + 1, 34) if A_karate[u, v] == 0]
    def _j(u, v):
        a, b = set(np.flatnonzero(_At[u])), set(np.flatnonzero(_At[v])); return len(a & b) / len(a | b) if a | b else 0.0
    _y = [1] * len(_hidden) + [0] * len(_neg)
    _s = [_j(u, v) for u, v in _hidden] + [_j(u, v) for u, v in _neg]
    check('AUC (vs sklearn)', _res['auc'], roc_auc_score(_y, _s), tol=1e-6)
    _pred = [(u, v) for u in range(34) for v in range(u + 1, 34) if _At[u, v] == 0 and _j(u, v) >= 0.2]
    _hits = len(set(_pred) & set((min(e), max(e)) for e in _hidden))
    check('precision', _res['precision'], _hits / len(_pred), tol=1e-6)
    check('recall', _res['recall'], _hits / 15, tol=1e-6)
else:
    check('link prediction', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Sample edge *indices*: `idx = r.choice(len(KARATE_EDGES), 15, replace=False)` (this is what the check does too).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

AUC $= \frac{1}{|P||N|}\sum_{p,n}[\mathbb 1(s_p > s_n) + \tfrac12\mathbb 1(s_p = s_n)]$; broadcast `pos[:, None]` against `neg[None, :]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Precision stays in the low single digits and varies a lot with the seed (just like the self-study's $1.6\%$–$4.2\%$), because the
positives are 15 out of ~500 candidates. The AUC is threshold-free and about 0.55–0.7 for Jaccard on karate (seeds 10, 42, 2026 give 0.67, 0.56, 0.62): better than random
(0.5) but far from perfect. Many hidden edges get score 0 (zero-similarity sparsity) and tie with thousands of negatives.

```python
def run_link_prediction(seed, threshold=0.2):
    r = np.random.default_rng(seed)
    idx = r.choice(len(KARATE_EDGES), 15, replace=False)
    hidden = [tuple(sorted(KARATE_EDGES[i])) for i in idx]
    A_train = A_karate.copy()
    for u, v in hidden:
        A_train[u, v] = A_train[v, u] = 0
    J = similarity_matrices(A_train)[1]
    iu, ju = np.triu_indices(34, 1)
    cand = A_train[iu, ju] == 0
    is_hidden = np.zeros((34, 34), bool)
    for u, v in hidden:
        is_hidden[u, v] = True
    pred = cand & (J[iu, ju] >= threshold)
    hits = (pred & is_hidden[iu, ju]).sum()
    precision, recall = hits / pred.sum(), hits / len(hidden)
    pos = np.array([J[u, v] for u, v in hidden])
    neg = J[iu, ju][A_karate[iu, ju] == 0]
    auc = np.mean((pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :]))
    return dict(precision=precision, recall=recall, auc=auc)

results_lp = {s: run_link_prediction(s) for s in (10, 42, 2026)}
for s, r_ in results_lp.items():
    print(s, {k: round(float(v), 3) for k, v in r_.items()})
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*